# Write the Costa 2024 ATST study

Combine both shared plate-reader workbooks into one study with two readouts (one per workbook, combining worksheets with matching metadata, assay and Time values). Follow the fully linked structure of simulated example e3.

Run from this folder, using a Python environment with ATST, pandas, openpyxl and matplotlib. The curated `layout_inputs/` and entity CSVs are inputs, with evidence documented in [curation.md](curation.md). They are not original author-provided plate layouts.

Readout partitions preserve source-workbook separation. `curve_id` identifies source columns. Original numeric values and rounded source times are retained.

In [1]:
from pathlib import Path

import warnings

import pandas as pd
from openpyxl.utils import get_column_letter

from ATST import ATSTFile, MultiReadoutATST, validate_atst, write_multi_readout_atst
from ATST.blocks import (
    Assay, Entities, EntityTable, FileMetadata, Layout, Metadata,
    Readings, ReadoutIds, Study,
)

# Keep the current ATST parser's repeated pandas fragmentation warning out of the walkthrough.
warnings.filterwarnings("ignore", message="DataFrame is highly fragmented", category=pd.errors.PerformanceWarning, module=r"ATST\.numeric")

root = Path.cwd()
workbooks = {"raw": "Plate_reader_raw.xlsx", "moi10": "Plate_reader_raw_MOI10.xlsx"}

## Study and file identity

In [2]:
import datetime


file_info = FileMetadata(data={
    "file_name": "atst_costa_2024.atst.txt",
    "format": "ATST", "format_version": "0.1",
    "created_on": datetime.datetime.now().date().isoformat(),
    "field_delimiter": "TAB", "encoding": "UTF-8",
})
study = Study(data={
    "title": "Accumulation of defense systems in phage-resistant strains of Pseudomonas aeruginosa",
    "study_id": "doi:10.1126/sciadv.adj0341",
    "authors": "Ana Rita Costa; Daan F. van den Berg; Jelger Q. Esser; Aswin Muralidharan; Halewijn van den Bossche; Boris Estrada Bonilla; Baltus A. van der Steen; Anna C. Haagsma; Ad C. Fluit; Franklin L. Nobrega; Pieter-Jan Haas; Stan J. J. Brouns",
    "publication_date": "2024-02-23",
    "publication_doi": "10.1126/sciadv.adj0341",
    "dataset_doi": "10.5281/zenodo.8405451",
    "dataset_version": "v1.0",
    "bioproject": "PRJNA817167",
    "description": "Liquid culture collapse assays of PAO1 carrying individual defense systems and an empty-vector comparator; both shared plate-reader workbooks.",
    "original_repository": "https://github.com/BrounsLab-TUDelft/Pseudomonas_Defence_Systems/tree/v1.0",
})

## Entity dictionaries and shared assay conditions

Entity CSVs contain material identities curated from Tables S3/S8 and the paper. Unknown accessions remain blank, with the author's `TBD` retained separately. Taxonomy is recorded as published, not updated to a current classification.

The assay fields below apply across the source sheets. The MOI10 layouts use `moi`; the raw-workbook layouts use `pfu_ml`. The raw `dilution 1` and `dilution 2` labels are assigned `1*10^7` and `1*10^6` from the labelled sheets; they are not converted to MOI or given concentration units.

In [3]:
entities = Entities(tables={
    name: EntityTable(
        name="ENTITIES", table_name=name, pk=pk,
        data=pd.read_csv(root / "entities" / f"{name}.csv", dtype=str, keep_default_na=False),
    )
    for name, pk in [("ISOLATES", "ISOLATE_ID"), ("PHAGES", "PHAGE_ID"), ("MEDIA", "MEDIUM_ID")]
})

assay_data = {
    "readout_type": "absorbance",
    "readout_unit": "od600",
    "time_unit": "h",
    "plate_format": "96_well",
    "wavelength_nm": "600",
    "temperature_c": "37",
    "shaking_mode": "double orbital",
    "measurement_interval_min": "10",
    "duration_h": "24",
    "medium_id": "LB",
    "starting_od600": "approximately 0.1",
    "inoculum_preparation": "Overnight-grown bacteria diluted in LB before phage addition",
    "replicates_per_condition": "3",
    "replicate_type": "biological (paper default; replicate numbers are local source-column order)",
    "culture_selection": "Carbenicillin 200 ug/mL for plasmid-bearing P. aeruginosa; final assay supplementation not explicitly restated",
    "source_reference": "Paper p. 9, Liquid culture collapse assays; p. 10, Statistical analysis; p. 7, Bacterial strains (culture selection)"
}

## Read source measurements and attach the curated layouts

Each layout has only six columns. Within each worksheet input, `curve_A` corresponds to Excel B, `curve_B` to Excel C, and so on. Merged readouts reassign unique curve IDs; source-sheet context remains in `curation.md` and the source files. Empty trailing spreadsheet rows are ignored; measurements are not averaged, interpolated, blank-subtracted or otherwise transformed. Triplicate indices describe column order within a source header group, not matched individuals across conditions.

In [4]:
readouts = {}
for prefix, filename in workbooks.items():
    with pd.ExcelFile(root / "sources" / "Plate reader" / filename, engine="openpyxl") as book:
        for sheet in book.sheet_names:
            readout_id = prefix + "_" + sheet.replace(" ", "_").replace("-", "_")
            raw = pd.read_excel(book, sheet_name=sheet, header=None, dtype=str, keep_default_na=False)
            raw = raw.loc[raw.ne("").any(axis=1)]
            layout_data = pd.read_csv(
                root / "layout_inputs" / f"layout_{readout_id}.tsv",
                sep="\t", dtype=str, keep_default_na=False,
            )
            # The first reading column is source Excel B, mapped to curve_A.
            assert layout_data["curve_id"].tolist() == [
                "curve_" + get_column_letter(i) for i in range(1, raw.shape[1])
            ]
            readings_data = raw.iloc[1:].copy().reset_index(drop=True)
            readings_data.columns = ["Time", *layout_data["curve_id"]]
            metadata = Metadata(data={
                "instrument": "Epoch2", "instrument_manufacturer": "BioTek",
                "instrument_source": "Paper p. 9, Liquid culture collapse assays",
                "source_file": f"sources/Plate reader/{filename}",
                "source_data_rows": f"2:{len(readings_data) + 1}",
                "source_dataset_doi": "10.5281/zenodo.8405451",
                "conversion": "Preserve numeric values and source times; map columns to curve_id using curated layout",
            })
            readouts[readout_id] = ATSTFile(
                file_info=file_info, study=study, metadata=metadata,
                assay=Assay(data=assay_data.copy()), entities=entities,
                layout=Layout(data=layout_data), readings=Readings(data=readings_data),
            )

def merged_name(members):
    return "raw_all" if members[0][0].startswith("raw_") else "moi10_all"

# Combine readouts only when metadata, assay, and recorded times match.
groups_by_context = {}
for readout_id, item in readouts.items():
    key = (tuple(item.metadata.data.items()), tuple(item.assay.data.items()), tuple(item.readings.data["Time"].astype(str)))
    groups_by_context.setdefault(key, []).append((readout_id, item))
combined = {}
for members in groups_by_context.values():
    if len(members) == 1:
        combined[members[0][0]] = members[0][1]
        continue
    layouts = []
    readings = members[0][1].readings.data[["Time"]].copy()
    for _, item in members:
        rename = {}
        for old_id in item.layout.data["curve_id"]:
            letter = get_column_letter(len(readings.columns) + len(rename))
            rename[old_id] = "curve_ID_238" if letter == "ID" else "curve_" + letter
        layout = item.layout.data.copy()
        layout["curve_id"] = layout["curve_id"].map(rename)
        layouts.append(layout)
        readings = pd.concat([readings, item.readings.data.drop(columns="Time").rename(columns=rename).reset_index(drop=True)], axis=1)
    first = members[0][1]
    combined[merged_name(members)] = ATSTFile(file_info=file_info, study=study, metadata=first.metadata, assay=first.assay, entities=entities, layout=Layout(data=pd.concat(layouts, ignore_index=True)), readings=Readings(data=readings))
readouts = combined

readout_ids = ReadoutIds(data=pd.DataFrame({"readout_id": list(readouts)}))
for readout in readouts.values():
    readout.readout_ids = readout_ids
atst = MultiReadoutATST(file_info, study, readout_ids, readouts, entities)
validate_atst(atst)
pd.DataFrame([
    {"readout_id": key, "curves": len(value.layout.data), "time_points": len(value.readings.data)}
    for key, value in readouts.items()
])

## Write the linked study

In [5]:
output = write_multi_readout_atst(atst, root.parent / 'atst_costa_2024', linked_files=True)

print(f"{len(readouts)} readouts; {sum(len(r.layout.data) for r in readouts.values())} curves")

2 readouts; 783 curves
